# Battery revenue stacking — walkthrough

This notebook runs the model end to end on the synthetic sample day: first
energy arbitrage alone, then arbitrage co-optimized with frequency
regulation capacity, and finally a sweep over the deployment fraction
`phi`.

**The price data is synthetic.** It is shaped to look like a plausible day
in a market with meaningful solar penetration, but it is not taken from or
calibrated against any real ISO. Nothing here is a claim about any actual
market — see `docs/formulation.md`.

In [ ]:
from pathlib import Path

from bess_opt.data.loaders import load_price_series_csv
from bess_opt.data.schema import Battery, System
from bess_opt.model.builder import build_bess_model
from bess_opt.solve import solve_bess
from bess_opt.viz import plot_price_and_dispatch, plot_revenue_stack, plot_soc_trajectory

SAMPLE_DIR = Path.cwd().parent / "data" / "sample_price_series"
prices = load_price_series_csv(SAMPLE_DIR / "day_hourly.csv")

print(f"{len(prices)} hourly periods")
print(f"energy price range: ${min(prices.energy):.2f} to ${max(prices.energy):.2f}/MWh")
print(f"reg-up price range: ${min(prices.reg_up):.2f} to ${max(prices.reg_up):.2f}/MW")

## The battery

A 10 MW / 40 MWh unit — a four-hour battery, which is the shape most grid
storage takes. Round-trip efficiency of about 88% (0.94 each way). It starts
half full, and the terminal constraint requires it to end that way too, so
the day is energy-neutral and the reported profit is not inflated by selling
off the starting charge.

In [ ]:
battery = Battery(
    name="Batt1",
    p_charge_max=10.0,
    p_discharge_max=10.0,
    energy_max=40.0,
    energy_min=0.0,
    initial_soc=20.0,
    charge_efficiency=0.94,
    discharge_efficiency=0.94,
)

print(f"usable energy: {battery.usable_energy} MWh")
print(f"round-trip efficiency: {battery.round_trip_efficiency:.1%}")

## 1. Arbitrage only

`include_regulation=False` pins the capacity variables to zero, so this is a
pure energy-arbitrage problem running through the same builder.

In [ ]:
arbitrage_system = System(battery=battery, prices=prices, include_regulation=False)
arbitrage_result = solve_bess(build_bess_model(arbitrage_system))

print(f"arbitrage-only profit: ${arbitrage_result.total_profit:,.2f}")

In [ ]:
plot_price_and_dispatch(arbitrage_system, arbitrage_result)

The battery buys through the overnight trough and the negative-price hour in
the middle of the day, then sells into the evening peak. The state of charge
shows the same story as a stock rather than a flow — and it returns to where
it started.

In [ ]:
plot_soc_trajectory(arbitrage_system, arbitrage_result)

## 2. Adding regulation capacity

Now the same battery, same day, with regulation capacity available. The
arbitrage-only schedule is still feasible here, so stacking can only add
value — the question is how much, and what it costs in arbitrage terms.

In [ ]:
stacked_system = System(battery=battery, prices=prices, phi=0.5)
stacked_result = solve_bess(build_bess_model(stacked_system))

uplift = stacked_result.total_profit - arbitrage_result.total_profit

print(f"arbitrage-only profit: ${arbitrage_result.total_profit:>10,.2f}")
print(f"stacked profit:        ${stacked_result.total_profit:>10,.2f}")
print(f"uplift:                ${uplift:>10,.2f}")
print()
for service, amount in stacked_result.revenue_breakdown().items():
    print(f"  {service:<12} ${amount:>10,.2f}")

Note what happens to the arbitrage component: it typically *falls* relative
to the arbitrage-only case. Holding power headroom and energy in reserve for
regulation is not free — the model gives up some energy margin because the
capacity payment more than covers it. That trade-off is the whole point of
co-optimizing rather than solving the two markets separately.

In [ ]:
plot_revenue_stack(stacked_system, stacked_result)

In [ ]:
plot_price_and_dispatch(stacked_system, stacked_result)

## 3. How much does the deployment assumption matter?

`phi` is the assumed fraction of committed capacity that actually gets
called. It sizes the SOC headroom the battery must hold back: at `phi=0`
capacity is free of any energy reservation, and at `phi=1` every committed
MW must be backed by a full period of energy.

**`phi` is an assumption, not a market rule.** Real ISO regulation products
have published mileage and performance-score rules that this model does not
implement. The sweep below shows how much the answer moves with the
assumption, which is the honest way to present a parameter like this.

In [ ]:
print(f"{'phi':>5}  {'profit':>12}  {'reg revenue':>12}  {'MW committed':>13}")
for phi in [0.0, 0.25, 0.5, 0.75, 1.0]:
    system = System(battery=battery, prices=prices, phi=phi)
    result = solve_bess(build_bess_model(system))
    committed = sum(result.reg_up.values()) + sum(result.reg_down.values())
    print(
        f"{phi:>5.2f}  ${result.total_profit:>11,.2f}  "
        f"${result.regulation_revenue:>11,.2f}  {committed:>13,.1f}"
    )

## What this model does not do

- **No degradation.** Cycling is free here, so the model will cycle harder
  than an operator watching warranty terms would.
- **Perfect foresight.** The whole price series is known up front. A real
  operator re-optimizes against a forecast as the day unfolds, and earns
  less than this number.
- **Price taker.** The battery's own bids are assumed not to move prices.
- **No specific ISO's rules.** `phi` is a simplification, not a tariff.

Full list with rationale in `docs/formulation.md`.